# Residual-gas background vs photon energy

Run 58890 is the residual-gas scan used as the photon-energy-dependent background for the config-1 glycine scans.

This notebook starts from the shot-resolved static-XAS file written by `compute_static_xas_cfg1.py`. The energy sections have already been detected from the local-DAQ train-ID starts, so the analysis here focuses on:

- averaging eTOF and iTOF spectra per photon energy,
- visualising the background as a 2D energy x TOF map,
- quantifying how the background intensity and shape change with photon energy,
- creating a per-energy background table that can be used for glycine subtraction,
- optionally loading a glycine signal map and plotting signal, background, and signal minus background.

In [ ]:
import sys
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

%matplotlib inline

# Locate repo root and put analysis/scripts on the import path.
cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd] + list(cwd.parents):
    if (p / "analysis" / "scripts").exists():
        repo_root = p
        break
if repo_root is None:
    raise RuntimeError("Could not find analysis/scripts in this repo.")
sys.path.insert(0, str(repo_root / "analysis" / "scripts"))

import config as path_config  # noqa: E402
from compute_aggregates import load_aggregates  # noqa: E402

## Inputs

`BACKGROUND_STATIC_H5` should already exist after running:

```bash
FLASH_ENV=remote python analysis/scripts/compute_static_xas_cfg1.py \
  analysis/configs/tof_static/background_run58890.py
```

For the subtraction plots, use either:

- `SIGNAL_STATIC_H5`: a compatible glycine static-XAS output, or
- `SIGNAL_AGG_FILES`: the aggregate files used by `cfg1_etof_vs_energy.ipynb`, one file per photon energy.

Leave both as `None` while developing the background-only part.

In [ ]:
RUN_NO = 58890

BACKGROUND_STATIC_H5 = (
    Path(path_config.COMBINED_DIR).parent
    / "xas_static"
    / f"run{RUN_NO}_cfg1_static_xas.h5"
)

# Option A: point this at a glycine static-XAS file with datasets matching
# compute_static_xas_cfg1.py output.
SIGNAL_STATIC_H5 = None
# Example:
# SIGNAL_STATIC_H5 = Path(path_config.COMBINED_DIR).parent / "xas_static" / "runXXXXX_cfg1_static_xas.h5"

# Option B: use config-1 aggregate files, as in cfg1_etof_vs_energy.ipynb.
# Each tuple is (path, photon_energy_eV). If SIGNAL_GMD_BIN is None, the
# notebook averages D/G across GMD bins weighted by n_per_bin.
SIGNAL_AGG_FILES = None
# Example:
# SIGNAL_AGG_FILES = [
#     (Path(path_config.COMBINED_DIR) / "glycine_WL_scan_271.0eV_aggregates.h5", 271.0),
#     (Path(path_config.COMBINED_DIR) / "glycine_WL_scan_271.5eV_aggregates.h5", 271.5),
# ]
SIGNAL_GMD_BIN = None

# TOF histogram edges. Use np.arange for a fixed bin width.
# The calibration notebook uses 1-unit eTOF bins over [0, 5000].
# Do not write np.linspace(0, 5000, 1): that creates only one edge,
# i.e. zero bins.
ETOF_BIN_WIDTH = 1.0
ETOF_BIN_EDGES = np.arange(0.0, 5000.0 + ETOF_BIN_WIDTH, ETOF_BIN_WIDTH)

# Ions are usually much later; keep a wider range by default.
ITOF_BIN_WIDTH = 20.0
ITOF_BIN_EDGES = np.arange(0.0, 40000.0 + ITOF_BIN_WIDTH, ITOF_BIN_WIDTH)

# Duplicate nominal energies: "keep_last" keeps the later repeat,
# "merge" averages repeats weighted by summed GMD, "keep_all" keeps sections.
DUPLICATE_ENERGY_POLICY = "keep_last"

# Optional scale factor for subtraction. Keep at 1.0 for GMD-normalised maps.
BACKGROUND_SCALE = 1.0

# Display ranges. Use (None, None) for full range.
ETOF_RANGE = (None, None)
ITOF_RANGE = (None, None)

print("background:", BACKGROUND_STATIC_H5)
print("signal static:", SIGNAL_STATIC_H5)
print("signal aggs  :", SIGNAL_AGG_FILES)

## Helpers

In [ ]:
def load_static_xas(path):
    """Load a compute_static_xas_cfg1.py output file."""
    path = Path(path)
    with h5py.File(path, "r") as f:
        out = {
            "path": path,
            "tofs_e": f["tofs_e"][...],
            "tofs_i": f["tofs_i"][...],
            "gmd": f["gmd"][...],
            "n_shots": f["n_shots"][...].astype(np.int64),
            "energies": f["nominal_energies"][...].astype(np.float64),
            "attrs": dict(f.attrs),
        }
    return out


def spectra_per_energy(tofs, gmd, n_shots, edges):
    """Histogram TOF hits per energy section and normalise by summed GMD."""
    n_energy = tofs.shape[0]
    n_bins = len(edges) - 1
    spec = np.full((n_energy, n_bins), np.nan, dtype=np.float64)
    sum_gmd = np.zeros(n_energy, dtype=np.float64)
    valid_shots = np.zeros(n_energy, dtype=np.int64)
    hit_counts = np.zeros(n_energy, dtype=np.int64)

    for i in range(n_energy):
        n = int(n_shots[i])
        if n <= 0:
            continue

        g = gmd[i, :n]
        ok = np.isfinite(g)
        if not ok.any():
            continue

        hits = tofs[i, :n][ok].ravel()
        hits = hits[hits > 0]
        sum_gmd[i] = float(np.nansum(g[ok]))
        valid_shots[i] = int(ok.sum())
        hit_counts[i] = int(hits.size)

        if hits.size == 0 or sum_gmd[i] <= 0:
            continue

        hist, _ = np.histogram(hits, bins=edges)
        spec[i] = hist / sum_gmd[i]

    return spec, sum_gmd, valid_shots, hit_counts


def merge_repeated_energies(spec, energies, weights=None):
    """Weighted-average spectra for repeated nominal energies."""
    energies = np.asarray(energies, dtype=np.float64)
    unique = np.array(sorted(np.unique(energies)))
    merged = np.full((unique.size, spec.shape[1]), np.nan, dtype=np.float64)
    if weights is None:
        weights = np.ones(energies.size, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)

    for i, e in enumerate(unique):
        idx = np.where(energies == e)[0]
        rows = spec[idx]
        w = weights[idx]
        valid = np.isfinite(rows).any(axis=1) & np.isfinite(w) & (w > 0)
        if not valid.any():
            continue
        merged[i] = np.average(rows[valid], axis=0, weights=w[valid])
    return unique, merged


def resolve_duplicate_energies(spec, energies, *, policy="keep_last", weights=None):
    """Handle repeated nominal energies after section-wise spectra are made."""
    energies = np.asarray(energies, dtype=np.float64)
    if policy == "keep_all":
        return energies, spec
    if policy == "merge":
        return merge_repeated_energies(spec, energies, weights=weights)
    if policy != "keep_last":
        raise ValueError("DUPLICATE_ENERGY_POLICY must be 'keep_last', 'merge', or 'keep_all'")

    keep = []
    for e in sorted(np.unique(energies)):
        idx = np.where(energies == e)[0]
        keep.append(idx[-1])
    keep = np.array(keep, dtype=np.int64)
    return energies[keep], spec[keep]


def half_step_edges(x):
    x = np.asarray(x, dtype=np.float64)
    if x.size == 1:
        return np.array([x[0] - 0.5, x[0] + 0.5])
    mid = 0.5 * (x[:-1] + x[1:])
    return np.concatenate([[2 * x[0] - mid[0]], mid, [2 * x[-1] - mid[-1]]])


def plot_map(spec, tof_edges, energies, *, title, clabel, tof_range=(None, None), cmap="viridis"):
    e_edges = half_step_edges(energies)
    finite = spec[np.isfinite(spec)]
    vmin = float(np.nanpercentile(finite, 1)) if finite.size else 0.0
    vmax = float(np.nanpercentile(finite, 99.5)) if finite.size else 1.0
    fig, ax = plt.subplots(figsize=(9, 4.8), constrained_layout=True)
    im = ax.pcolormesh(tof_edges, e_edges, spec, shading="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    fig.colorbar(im, ax=ax, label=clabel)
    ax.set_xlabel("TOF (100 ps ticks)")
    ax.set_ylabel("nominal photon energy (eV)")
    ax.set_title(title)
    if tof_range[0] is not None or tof_range[1] is not None:
        ax.set_xlim(tof_range)
    return fig, ax


def plot_diff_map(diff, tof_edges, energies, *, title, clabel, tof_range=(None, None)):
    e_edges = half_step_edges(energies)
    finite = diff[np.isfinite(diff)]
    lim = float(np.nanpercentile(np.abs(finite), 99.0)) if finite.size else 1.0
    if not np.isfinite(lim) or lim <= 0:
        lim = 1.0
    fig, ax = plt.subplots(figsize=(9, 4.8), constrained_layout=True)
    norm = TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim)
    im = ax.pcolormesh(tof_edges, e_edges, diff, shading="auto", cmap="RdBu_r", norm=norm)
    fig.colorbar(im, ax=ax, label=clabel)
    ax.set_xlabel("TOF (100 ps ticks)")
    ax.set_ylabel("nominal photon energy (eV)")
    ax.set_title(title)
    if tof_range[0] is not None or tof_range[1] is not None:
        ax.set_xlim(tof_range)
    return fig, ax

In [ ]:
def load_signal_aggregate_map(files, *, gmd_bin=None):
    """Load config-1 spectral aggregates into an energy x eTOF map."""
    files = sorted([(Path(p), float(e)) for p, e in files], key=lambda pe: pe[1])
    energies = np.array([e for _, e in files], dtype=np.float64)
    stack = []
    tof_edges = None
    shot_counts = []

    for path, energy in files:
        agg = load_aggregates(path)
        if agg.config != 1:
            raise ValueError(f"{path.name}: expected config 1, found config {agg.config}")
        if agg.mode != "spectral":
            raise ValueError(f"{path.name}: expected spectral aggregate, found {agg.mode!r}")
        if tof_edges is None:
            tof_edges = agg.tof_edges
        elif not np.array_equal(tof_edges, agg.tof_edges):
            raise ValueError(f"{path.name}: eTOF edges do not match previous aggregate files")

        with np.errstate(invalid="ignore", divide="ignore"):
            per_gmd = agg.D / agg.G[:, None]

        if gmd_bin is None:
            weights = np.asarray(agg.n_per_bin, dtype=np.float64)
            valid = np.isfinite(per_gmd).any(axis=1) & (weights > 0)
            if not valid.any():
                spec = np.full(per_gmd.shape[1], np.nan)
            else:
                spec = np.average(per_gmd[valid], axis=0, weights=weights[valid])
        else:
            spec = per_gmd[int(gmd_bin)]

        stack.append(spec)
        shot_counts.append(int(np.sum(agg.n_per_bin)))
        print(f"{energy:7.2f} eV  {path.name}  shots={shot_counts[-1]}")

    return {
        "energies": energies,
        "tof_edges": tof_edges,
        "spec_e": np.vstack(stack),
        "shot_counts": np.array(shot_counts, dtype=np.int64),
    }


def rebin_background_if_needed(bg_source, source_edges, target_edges):
    """Return background on target edges; currently requires identical bins."""
    if not np.array_equal(source_edges, target_edges):
        raise ValueError(
            "background and signal TOF edges differ. Re-run this notebook with "
            "ETOF_BIN_EDGES matching the aggregate TOF edges, or add explicit rebinning."
        )
    return bg_source

## Load and average run 58890

In [ ]:
bg = load_static_xas(BACKGROUND_STATIC_H5)

bg_spec_e_raw, bg_sum_gmd, bg_valid_shots, bg_hits_e = spectra_per_energy(
    bg["tofs_e"], bg["gmd"], bg["n_shots"], ETOF_BIN_EDGES
)
bg_spec_i_raw, _, _, bg_hits_i = spectra_per_energy(
    bg["tofs_i"], bg["gmd"], bg["n_shots"], ITOF_BIN_EDGES
)

bg_energies, bg_spec_e = resolve_duplicate_energies(
    bg_spec_e_raw, bg["energies"], policy=DUPLICATE_ENERGY_POLICY, weights=bg_sum_gmd
)
_, bg_spec_i = resolve_duplicate_energies(
    bg_spec_i_raw, bg["energies"], policy=DUPLICATE_ENERGY_POLICY, weights=bg_sum_gmd
)

print(f"loaded {bg['path'].name}")
print(f"sections              : {bg['energies'].size}")
print(f"plotted energies       : {bg_energies.size} ({DUPLICATE_ENERGY_POLICY})")
print(f"shots total            : {int(bg['n_shots'].sum())}")
print(f"signal bunch range attr: {tuple(bg['attrs'].get('signal_bunch_range', []))}")
print(f"section source attr    : {bg['attrs'].get('section_source', 'not recorded')}")

for i, e in enumerate(bg["energies"]):
    print(
        f"section {i:2d}  E={e:6.2f} eV  "
        f"shots={int(bg['n_shots'][i]):7d}  "
        f"sumGMD={bg_sum_gmd[i]:10.3g}  "
        f"eHits={bg_hits_e[i]:8d}  iHits={bg_hits_i[i]:8d}"
    )

## Background 2D maps

In [ ]:
plot_map(
    bg_spec_e,
    ETOF_BIN_EDGES,
    bg_energies,
    title="Residual-gas eTOF background vs photon energy",
    clabel="eTOF hits / summed GMD / bin",
    tof_range=ETOF_RANGE,
)
plt.show()

plot_map(
    bg_spec_i,
    ITOF_BIN_EDGES,
    bg_energies,
    title="Residual-gas iTOF background vs photon energy",
    clabel="iTOF hits / summed GMD / bin",
    tof_range=ITOF_RANGE,
)
plt.show()

## Intensity and shape scaling

The integrated intensity tracks the total residual-gas yield per GMD. The shape metric compares each spectrum to the mean background shape after area normalisation.

In [ ]:
def spectrum_area(spec, edges):
    # Spectra are histogram counts per summed GMD per bin, not densities.
    # The total yield is therefore the sum over bins.
    return np.nansum(spec, axis=1)


def shape_distance(spec, edges):
    area = spectrum_area(spec, edges)
    shape = spec / area[:, None]
    ref = np.nanmean(shape, axis=0)
    return np.sqrt(np.nansum((shape - ref) ** 2, axis=1))


e_area = spectrum_area(bg_spec_e, ETOF_BIN_EDGES)
i_area = spectrum_area(bg_spec_i, ITOF_BIN_EDGES)
e_shape_dist = shape_distance(bg_spec_e, ETOF_BIN_EDGES)
i_shape_dist = shape_distance(bg_spec_i, ITOF_BIN_EDGES)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
axes[0].plot(bg_energies, e_area, "o-", label="eTOF")
axes[0].plot(bg_energies, i_area, "o-", label="iTOF")
axes[0].set_xlabel("photon energy (eV)")
axes[0].set_ylabel("integrated hits / GMD")
axes[0].set_title("Background intensity")
axes[0].legend()

axes[1].plot(bg_energies, e_shape_dist, "o-", label="eTOF")
axes[1].plot(bg_energies, i_shape_dist, "o-", label="iTOF")
axes[1].set_xlabel("photon energy (eV)")
axes[1].set_ylabel("distance from mean shape")
axes[1].set_title("Background shape variation")
axes[1].legend()
plt.show()

## Save the per-energy background product

This creates a compact H5 file with the merged, GMD-normalised background spectra. Downstream glycine notebooks can interpolate this onto their energy axis and subtract by matching TOF bins.

In [ ]:
OUT_H5 = Path(path_config.COMBINED_DIR).parent / "xas_static" / f"run{RUN_NO}_cfg1_background_per_energy.h5"

with h5py.File(OUT_H5, "w") as f:
    f.create_dataset("energies", data=bg_energies)
    f.create_dataset("etof_edges", data=ETOF_BIN_EDGES)
    f.create_dataset("itof_edges", data=ITOF_BIN_EDGES)
    f.create_dataset("etof_background", data=bg_spec_e)
    f.create_dataset("itof_background", data=bg_spec_i)
    f.create_dataset("section_energies", data=bg["energies"])
    f.create_dataset("section_n_shots", data=bg["n_shots"])
    f.create_dataset("section_sum_gmd", data=bg_sum_gmd)
    f.attrs["source_static_xas"] = str(BACKGROUND_STATIC_H5)
    f.attrs["normalisation"] = "hits per summed GMD per TOF bin"
    f.attrs["duplicate_energy_policy"] = DUPLICATE_ENERGY_POLICY

print("wrote", OUT_H5)

## Optional glycine subtraction

Set `SIGNAL_STATIC_H5` above to a glycine static-XAS file with compatible TOF binning. This section plots the glycine signal map, the residual-gas background interpolated onto the glycine photon-energy axis, and the difference.

In [ ]:
if SIGNAL_STATIC_H5 is not None:
    sig = load_static_xas(SIGNAL_STATIC_H5)
    sig_spec_e_raw, sig_sum_gmd, _, _ = spectra_per_energy(
        sig["tofs_e"], sig["gmd"], sig["n_shots"], ETOF_BIN_EDGES
    )
    sig_spec_i_raw, _, _, _ = spectra_per_energy(
        sig["tofs_i"], sig["gmd"], sig["n_shots"], ITOF_BIN_EDGES
    )

    sig_energies, sig_spec_e = resolve_duplicate_energies(
        sig_spec_e_raw, sig["energies"], policy=DUPLICATE_ENERGY_POLICY, weights=sig_sum_gmd
    )
    _, sig_spec_i = resolve_duplicate_energies(
        sig_spec_i_raw, sig["energies"], policy=DUPLICATE_ENERGY_POLICY, weights=sig_sum_gmd
    )

    bg_on_sig_e = np.vstack([
        np.interp(sig_energies, bg_energies, bg_spec_e[:, j], left=np.nan, right=np.nan)
        for j in range(bg_spec_e.shape[1])
    ]).T
    bg_on_sig_i = np.vstack([
        np.interp(sig_energies, bg_energies, bg_spec_i[:, j], left=np.nan, right=np.nan)
        for j in range(bg_spec_i.shape[1])
    ]).T

    sub_e = sig_spec_e - BACKGROUND_SCALE * bg_on_sig_e
    sub_i = sig_spec_i - BACKGROUND_SCALE * bg_on_sig_i

    plot_map(sig_spec_e, ETOF_BIN_EDGES, sig_energies, title="Glycine eTOF signal", clabel="hits / GMD / bin", tof_range=ETOF_RANGE)
    plt.show()
    plot_map(bg_on_sig_e, ETOF_BIN_EDGES, sig_energies, title="Residual-gas eTOF background on glycine energy axis", clabel="hits / GMD / bin", tof_range=ETOF_RANGE)
    plt.show()
    plot_diff_map(sub_e, ETOF_BIN_EDGES, sig_energies, title="Glycine eTOF minus residual-gas background", clabel="subtracted hits / GMD / bin", tof_range=ETOF_RANGE)
    plt.show()

    plot_map(sig_spec_i, ITOF_BIN_EDGES, sig_energies, title="Glycine iTOF signal", clabel="hits / GMD / bin", tof_range=ITOF_RANGE)
    plt.show()
    plot_map(bg_on_sig_i, ITOF_BIN_EDGES, sig_energies, title="Residual-gas iTOF background on glycine energy axis", clabel="hits / GMD / bin", tof_range=ITOF_RANGE)
    plt.show()
    plot_diff_map(sub_i, ITOF_BIN_EDGES, sig_energies, title="Glycine iTOF minus residual-gas background", clabel="subtracted hits / GMD / bin", tof_range=ITOF_RANGE)
    plt.show()

elif SIGNAL_AGG_FILES is not None:
    sig = load_signal_aggregate_map(SIGNAL_AGG_FILES, gmd_bin=SIGNAL_GMD_BIN)
    sig_energies = sig["energies"]
    sig_spec_e = sig["spec_e"]
    sig_edges = sig["tof_edges"]

    bg_spec_for_signal = rebin_background_if_needed(bg_spec_e, ETOF_BIN_EDGES, sig_edges)
    bg_on_sig_e = np.vstack([
        np.interp(sig_energies, bg_energies, bg_spec_for_signal[:, j], left=np.nan, right=np.nan)
        for j in range(bg_spec_for_signal.shape[1])
    ]).T
    sub_e = sig_spec_e - BACKGROUND_SCALE * bg_on_sig_e

    suffix = "weighted over GMD bins" if SIGNAL_GMD_BIN is None else f"GMD bin {SIGNAL_GMD_BIN}"
    plot_map(sig_spec_e, sig_edges, sig_energies, title=f"Glycine aggregate eTOF signal ({suffix})", clabel="D / G", tof_range=ETOF_RANGE)
    plt.show()
    plot_map(bg_on_sig_e, sig_edges, sig_energies, title="Residual-gas eTOF background on glycine aggregate energy axis", clabel="hits / GMD / bin", tof_range=ETOF_RANGE)
    plt.show()
    plot_diff_map(sub_e, sig_edges, sig_energies, title="Glycine aggregate eTOF minus residual-gas background", clabel="subtracted D / G", tof_range=ETOF_RANGE)
    plt.show()

else:
    print("Set SIGNAL_STATIC_H5 or SIGNAL_AGG_FILES to enable glycine subtraction plots.")